# Predicting Customer Churn
### A Data-Driven Analysis of Subscription Customer Behaviour
**IDRA Data Science & AI Capstone 2026 — Project 4 (Classification, Target: Churn)**

---
**How to use this notebook.** Each section maps to a section of your report.
Read the comments *before* running a cell — they explain *why* each step is
done, so you can explain it in your report and viva. Run the cells top to
bottom (Runtime → Run all in Colab). Key charts are saved to a `figures/`
folder so you can insert them into the PDF report.


## 1. Imports & setup
We load only what we need: pandas/numpy for data, matplotlib/seaborn for
charts, and the scikit-learn pieces for the model and its evaluation.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, accuracy_score)

# Make plots consistent and readable for the report
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
os.makedirs("figures", exist_ok=True)   # charts for the report land here
RANDOM_STATE = 42                        # fixed seed = reproducible results

## 2. Load the data & first look
Report section: **Dataset Description**.
We check the shape (rows, columns), the data types, and a quick statistical
summary. Note two things straight away: `TotalCharges` shows up as an
*object* (text) even though it is money, and `customerID` is just a unique
label.

In [ ]:
# Update this path if your filename differs
df = pd.read_csv("P_4_WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Shape (rows, columns):", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Target balance — how many customers actually churned?
print(df["Churn"].value_counts())
print()
print(df["Churn"].value_counts(normalize=True).round(3))
# ~26.5% churn vs ~73.5% stay -> the classes are IMBALANCED.
# Remember this: a model can score 73% accuracy by predicting "No" for
# everyone. That is why we will judge it on precision / recall / F1, not
# accuracy alone.

## 3. Data cleaning
Report section: **Data Cleaning**. Every decision below has a stated reason —
that is exactly what the guide asks for (§7 / §5.2).

Three issues to handle:
1. `customerID` — a unique identifier, no predictive value → drop it.
2. `TotalCharges` — stored as text and has blank entries → convert & fix.
3. `SeniorCitizen` — stored as 0/1 but is really a category → relabel.

In [ ]:
# --- Issue 1: drop the ID column (a unique label cannot help prediction) ---
df = df.drop(columns=["customerID"])

# --- Issue 2: TotalCharges is text; force it to numeric ---
# errors="coerce" turns anything non-numeric (the blanks) into NaN so we
# can see them clearly.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

missing = df["TotalCharges"].isna().sum()
print("Blank TotalCharges values:", missing)
# Look at WHO these customers are before deciding what to do:
df.loc[df["TotalCharges"].isna(), ["tenure", "MonthlyCharges", "TotalCharges"]]

In [ ]:
# The blanks are ALL customers with tenure = 0 (brand new, never billed a
# full cycle). Total charges of 0 is the logical value for them, so we fill
# with 0 rather than dropping the rows.
# (Reason to fill, not drop: the guide warns against dropping rows blindly;
#  these are valid customers, and 0 is meaningful, not a guess.)
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# --- Issue 3: SeniorCitizen 0/1 -> readable category for EDA ---
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

# --- Validation: prove the problems are fixed (guide asks for this) ---
print("Remaining missing values in whole dataset:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("TotalCharges dtype now:", df["TotalCharges"].dtype)

## 4. Exploratory Data Analysis
Report sections: **EDA**, **Statistical Analysis**, **Visualisation**.
We only make charts that answer a question. For each one, write one line in
your report: *what it shows* and *why it matters for churn*.

### 4.1 How balanced is churn? (Figure 1)

In [ ]:
plt.figure(figsize=(5,4))
ax = sns.countplot(data=df, x="Churn", hue="Churn", palette="Set2", legend=False)
plt.title("Figure 1. Churn distribution")
plt.ylabel("Number of customers")
for c in ax.containers:
    ax.bar_label(c)
plt.tight_layout()
plt.savefig("figures/fig1_churn_balance.png", bbox_inches="tight")
plt.show()
# INSIGHT: churners are the minority (~1 in 4). Keep this in mind at
# evaluation time.

### 4.2 Numeric variables: tenure, MonthlyCharges, TotalCharges (Figure 2)

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
print(df[num_cols].describe().round(2))   # mean, median (50%), spread, IQR

fig, axes = plt.subplots(1, 3, figsize=(14,4))
for ax, col in zip(axes, num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="#4C72B0")
    ax.set_title(col)
fig.suptitle("Figure 2. Distributions of numeric variables")
plt.tight_layout()
plt.savefig("figures/fig2_numeric_distributions.png", bbox_inches="tight")
plt.show()
# INSIGHT: tenure and TotalCharges are skewed with a big spike at the low
# end -> lots of short-tenure customers. Where a distribution is skewed the
# median is a more honest 'centre' than the mean (guide §6.3).

### 4.3 Does contract type relate to churn? (Figure 3)

In [ ]:
# Churn RATE within each contract type (proportion, not raw count)
rate = (df.groupby("Contract")["Churn"]
          .apply(lambda s: (s == "Yes").mean())
          .sort_values(ascending=False))
print((rate*100).round(1))

plt.figure(figsize=(6,4))
sns.barplot(x=rate.index, y=rate.values, hue=rate.index,
            palette="Set2", legend=False)
plt.ylabel("Churn rate")
plt.title("Figure 3. Churn rate by contract type")
plt.tight_layout()
plt.savefig("figures/fig3_churn_by_contract.png", bbox_inches="tight")
plt.show()
# INSIGHT: month-to-month customers churn far more than one/two-year
# contract customers. Contract length looks like a strong predictor.

### 4.4 Does tenure relate to churn? (Figure 4)

In [ ]:
# Bucket tenure into ranges, then look at churn rate per bucket
bins = [0, 12, 24, 48, 72]
labels = ["0-12m", "13-24m", "25-48m", "49-72m"]
df["tenure_group"] = pd.cut(df["tenure"], bins=bins, labels=labels,
                            include_lowest=True)
trate = df.groupby("tenure_group", observed=True)["Churn"].apply(
        lambda s: (s == "Yes").mean())
print((trate*100).round(1))

plt.figure(figsize=(6,4))
sns.barplot(x=trate.index, y=trate.values, hue=trate.index,
            palette="Set2", legend=False)
plt.ylabel("Churn rate")
plt.title("Figure 4. Churn rate by tenure group")
plt.tight_layout()
plt.savefig("figures/fig4_churn_by_tenure.png", bbox_inches="tight")
plt.show()
# INSIGHT: churn is highest for the newest customers and falls as tenure
# grows -> early-lifecycle retention is where the risk is.
df = df.drop(columns=["tenure_group"])   # helper only; not a model feature

### 4.5 Correlation between numeric variables (Figure 5)

In [ ]:
plt.figure(figsize=(5,4))
sns.heatmap(df[num_cols].corr(), annot=True, cmap="Blues", fmt=".2f")
plt.title("Figure 5. Correlation among numeric variables")
plt.tight_layout()
plt.savefig("figures/fig5_correlation.png", bbox_inches="tight")
plt.show()
# INSIGHT: tenure and TotalCharges are strongly correlated (longer stay =>
# more billed over time). Correlation is NOT causation (guide §6.3).

## 5. Preprocessing
Report sections: **Data Preprocessing**, **ML Methodology**.
We turn the target and features into numbers, split into train/test, and
scale — in that order. The order matters: we fit the scaler on the training
data only, so no information from the test set leaks into training (§5.3).

In [ ]:
# --- Target: Churn Yes/No -> 1/0 ---
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

# --- Features: separate target (y) from predictors (X) ---
y = df["Churn"]
X = df.drop(columns=["Churn"])

# --- Encode categoricals with one-hot encoding ---
# get_dummies expands each category column into 0/1 columns. drop_first=True
# avoids redundant columns. Numeric columns pass through untouched.
X = pd.get_dummies(X, drop_first=True)
print("Feature matrix shape after encoding:", X.shape)
X.head()

In [ ]:
# --- Train / test split BEFORE scaling ---
# stratify=y keeps the ~26.5% churn ratio in both train and test sets, which
# matters for imbalanced data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)

# --- Scale numeric columns (fit on TRAIN only -> apply to TEST) ---
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])   # learn + apply
X_test[num_cols]  = scaler.transform(X_test[num_cols])        # apply only
print("Scaling done (no leakage: scaler learned only from training data).")

## 6. Model training
Report section: **ML Methodology**.
Task = **classification** (Churn / Stay). We train two models:

- **Logistic Regression** — simple, fast, and its coefficients tell us
  *which factors push churn up or down* (great for the Discussion).
- **Random Forest** — a stronger model for comparison, and it gives feature
  importances.

Only keep both in your final report if you understand both.

In [ ]:
# Model 1: Logistic Regression
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train, y_train)

# Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

print("Both models trained.")

## 7. Model results & evaluation
Report section: **Model Results & Evaluation**.
For classification the guide requires: a **confusion matrix** plus
**accuracy, precision, recall and F1** — and an explanation of what false
positives / false negatives mean *for this problem*.

In churn terms:
- **False negative** = we predict "stay" but the customer actually churns →
  we miss an at-risk customer (usually the costlier mistake).
- **False positive** = we predict "churn" but they stay → we spend a
  retention offer on someone who was fine.

In [ ]:
def evaluate(model, name):
    pred = model.predict(X_test)
    print(f"===== {name} =====")
    print(classification_report(y_test, pred,
          target_names=["Stay (0)", "Churn (1)"]))
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Stay", "Churn"])
    disp.plot(cmap="Blues", colorbar=False)
    plt.title(f"Confusion matrix — {name}")
    plt.tight_layout()
    fname = f"figures/cm_{name.lower().replace(' ','_')}.png"
    plt.savefig(fname, bbox_inches="tight")
    plt.show()
    return pred

_ = evaluate(logreg, "Logistic Regression")

In [ ]:
_ = evaluate(rf, "Random Forest")

In [ ]:
# --- Overfitting check: compare train vs test accuracy (guide §9.4) ---
for name, model in [("Logistic Regression", logreg), ("Random Forest", rf)]:
    tr = accuracy_score(y_train, model.predict(X_train))
    te = accuracy_score(y_test,  model.predict(X_test))
    print(f"{name:22s} train={tr:.3f}  test={te:.3f}  gap={tr-te:.3f}")
# A large train>>test gap suggests overfitting. Logistic Regression usually
# shows a small gap; Random Forest a larger one. Discuss what you see.

## 8. What drives churn?
Report section: **Discussion**. This links the model back to the business
problem — which factors matter, and in which direction.

In [ ]:
# Logistic Regression coefficients: positive => pushes churn UP.
coef = pd.Series(logreg.coef_[0], index=X_train.columns).sort_values()
top = pd.concat([coef.head(6), coef.tail(6)])

plt.figure(figsize=(7,5))
colors = ["#C44E52" if v > 0 else "#4C72B0" for v in top.values]
sns.barplot(x=top.values, y=top.index, palette=colors, hue=top.index, legend=False)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Figure 6. Strongest churn drivers (Logistic Regression)")
plt.xlabel("Coefficient  (red = raises churn, blue = lowers churn)")
plt.tight_layout()
plt.savefig("figures/fig6_churn_drivers.png", bbox_inches="tight")
plt.show()

In [ ]:
# Random Forest feature importances (magnitude only, no direction)
imp = pd.Series(rf.feature_importances_, index=X_train.columns
               ).sort_values(ascending=False).head(10)
plt.figure(figsize=(7,5))
sns.barplot(x=imp.values, y=imp.index, hue=imp.index,
            palette="viridis", legend=False)
plt.title("Figure 7. Top 10 features by importance (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("figures/fig7_feature_importance.png", bbox_inches="tight")
plt.show()

## 9. Save the cleaned dataset
The guide asks you to submit the cleaned/preprocessed data alongside the
notebook.

In [ ]:
df.to_csv("telco_churn_cleaned.csv", index=False)
print("Saved: telco_churn_cleaned.csv")
print("Saved figures:", os.listdir("figures"))

---
## From here → your report
You now have the evidence. Write these report sections **in your own words**:

- **Abstract / Introduction** — the churn problem and why it matters.
- **EDA & Visualisation** — use Figures 1-5; one insight line per figure.
- **Model Results** — the confusion matrices + the precision/recall/F1 table.
- **Discussion** — use Figures 6-7: contract type, tenure, and internet
  service as the main churn drivers.
- **Limitations** — no causal proof, single telecom snapshot, class imbalance.
- **Recommendations** — e.g. *short-tenure + month-to-month customers churn
  most → target a first-3-months retention programme.*

Remember the integrity rule: be ready to explain any cell you keep.